# Terragrunt blocks

In [1]:
from bs4 import BeautifulSoup, NavigableString
from os import path, environ
from IPython.display import display
from IPython.core.display import HTML
import re
import pandas as pd

assets_abspath = path.join(environ.get("ASSETS_ABSPATH", "-"), "terragrunt")
artifacts_abspath = path.join(
    environ.get("ARTIFACTS_ABSPATH", "-"), "terragrunt"
)
blocks_source_abspath = path.join(assets_abspath, "attributes.html")
blocks_target_abspath = path.join(artifacts_abspath, "attributes.csv")

In [2]:
with open(blocks_source_abspath, "r") as f:
    html = f.read()

In [8]:
DOMAIN = "https://terragrunt.gruntwork.io"
ROUTE = "/docs/reference/config-blocks-and-attributes"

soup = BeautifulSoup(html, "html.parser")
# Remove classes and styles
for elem in soup.descendants:
    if hasattr(elem, "class"):
        del elem["class"]
    if hasattr(elem, "style"):
        del elem["style"]

# Fix link bases
for anchor in soup.find_all("a"):
    if anchor["href"].startswith("#"):
        anchor["href"] = f"{DOMAIN}{ROUTE}{anchor['href']}"
    if anchor["href"].startswith("/"):
        anchor["href"] = f"{DOMAIN}{anchor['href']}"

# Split content
entries = []
current = None
for elem in soup.children:
    if isinstance(elem, NavigableString):
        continue
    if elem.name == "h3":
        if current is not None:
            entries.append(current)
        current = {
            "header_h3_soup": elem,
            "description_soup": BeautifulSoup(
                f"<h2>{elem.text}</h2>", "html.parser"
            ),
        }
        continue
    if current is not None:
        current["description_soup"].append(elem)
entries.append(current)

# Enrich entries
enriched = []
for entry in entries:
    header_text = entry["header_h3_soup"].text.strip()
    summary_text = entry["description_soup"].p.text.strip()
    summary_text = re.split(r"\.\s", summary_text)[0] + "."
    summary_text = re.sub(header_text, "___", summary_text)
    enriched.append(
        {
            **entry,
            "header_p_soup": BeautifulSoup(
                f"<p>{header_text}</p>", "html.parser"
            ),
            "summary_soup": BeautifulSoup(
                f"<p>{summary_text}</p>", "html.parser"
            ),
        }
    )

htmlful = []
for item in enriched:
    htmlful.append(
        {
            "header_p_html": item["header_p_soup"].prettify(),
            "summary_html": item["summary_soup"].prettify(),
            "description_html": item["description_soup"].prettify(),
        }
    )

In [9]:
df = pd.DataFrame(htmlful)
df.rename(
    columns={
        "header_p_html": "HeaderHtml",
        "summary_html": "SummaryHtml",
        "description_html": "DescriptionHtml",
    },
    inplace=True,
)
df["Tags"] = "WebScraped Terragrunt-v0.62.1"
df

,HeaderHtml,SummaryHtml,DescriptionHtml,Tags
0,<p>\n inputs\n</p>\n,<p>\n The ___ attribute is a map that is used ...,<h2>\n inputs\n</h2>\n<p>\n The\n <code>\n in...,WebScraped Terragrunt-v0.62.1
1,<p>\n download_dir\n</p>\n,<p>\n The terragrunt ___ string option can be ...,<h2>\n download_dir\n</h2>\n<p>\n The terragru...,WebScraped Terragrunt-v0.62.1
2,<p>\n prevent_destroy\n</p>\n,<p>\n Terragrunt ___ boolean flag allows you t...,<h2>\n prevent_destroy\n</h2>\n<p>\n Terragrun...,WebScraped Terragrunt-v0.62.1
3,<p>\n skip\n</p>\n,<p>\n The terragrunt ___ boolean flag can be u...,<h2>\n skip\n</h2>\n<p>\n The terragrunt\n <co...,WebScraped Terragrunt-v0.62.1
4,<p>\n iam_role\n</p>\n,<p>\n The ___ attribute can be used to specify...,<h2>\n iam_role\n</h2>\n<p>\n The\n <code>\n ...,WebScraped Terragrunt-v0.62.1
5,<p>\n iam_assume_role_duration\n</p>\n,<p>\n The ___ attribute can be used to\n spec...,<h2>\n iam_assume_role_duration\n</h2>\n<p>\n ...,WebScraped Terragrunt-v0.62.1
6,<p>\n iam_assume_role_session_name\n</p>\n,<p>\n The ___ attribute can be used\n to spec...,<h2>\n iam_assume_role_session_name\n</h2>\n<p...,WebScraped Terragrunt-v0.62.1
7,<p>\n iam_web_identity_token\n</p>\n,<p>\n The ___ attribute can be used along\n w...,<h2>\n iam_web_identity_token\n</h2>\n<p>\n Th...,WebScraped Terragrunt-v0.62.1
8,<p>\n terraform_binary\n</p>\n,<p>\n The terragrunt ___ string option can be\...,<h2>\n terraform_binary\n</h2>\n<p>\n The terr...,WebScraped Terragrunt-v0.62.1
9,<p>\n terraform_version_constraint\n</p>\n,<p>\n The terragrunt ___ string\n overrides t...,<h2>\n terraform_version_constraint\n</h2>\n<p...,WebScraped Terragrunt-v0.62.1


In [10]:
df.to_csv(blocks_target_abspath, sep="|", header=False, index=False)